# 05. Machine Learning Pipeline for Stress Detection

This notebook demonstrates the complete ML pipeline for physiological stress detection,
implementing LOSO (Leave-One-Subject-Out) cross-validation as the gold standard for
subject-independent evaluation.

## Table of Contents
1. [Setup and Data Loading](#1-setup-and-data-loading)
2. [Baseline Model Evaluation](#2-baseline-model-evaluation)
3. [Cross-Validation Analysis](#3-cross-validation-analysis)
4. [Hyperparameter Tuning](#4-hyperparameter-tuning)
5. [Ensemble Methods](#5-ensemble-methods)
6. [Model Comparison](#6-model-comparison)
7. [Final Evaluation](#7-final-evaluation)
8. [Results Summary](#8-results-summary)

## 1. Setup and Data Loading

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

# Import CalmSense ML modules
from src.models.ml import (
    LogisticRegressionClassifier,
    SVMClassifier,
    RandomForestClassifier,
    XGBoostClassifier,
    LightGBMClassifier,
    CatBoostClassifier,
    CrossValidator,
    HyperparameterTuner,
    StackingEnsemble,
    VotingEnsemble,
    ModelEvaluator,
    ImbalanceHandler,
    MLTrainingPipeline,
    get_classifier,
)

print("CalmSense ML Pipeline loaded successfully!")

In [ ]:
# Generate synthetic data (replace with real WESAD data loading)
np.random.seed(42)

N_SUBJECTS = 15
SAMPLES_PER_SUBJECT = 200
N_FEATURES = 60  # Matching our feature extraction pipeline
N_CLASSES = 3  # baseline, stress, amusement

def generate_wesad_like_data(n_subjects, samples_per_subject, n_features, n_classes):
    """Generate synthetic physiological data mimicking WESAD structure."""
    X_list, y_list, groups_list = [], [], []
    
    for subject_id in range(n_subjects):
        # Each subject has slightly different feature distributions
        subject_offset = np.random.randn(n_features) * 0.5
        
        for cls in range(n_classes):
            # Baseline: 40%, Stress: 35%, Amusement: 25%
            class_samples = int(samples_per_subject * [0.4, 0.35, 0.25][cls])
            
            # Class-specific patterns
            class_offset = np.random.randn(n_features) * cls * 0.3
            
            X_class = np.random.randn(class_samples, n_features) + subject_offset + class_offset
            y_class = np.full(class_samples, cls)
            groups_class = np.full(class_samples, subject_id)
            
            X_list.append(X_class)
            y_list.append(y_class)
            groups_list.append(groups_class)
    
    return np.vstack(X_list), np.concatenate(y_list), np.concatenate(groups_list)

# Generate data
X, y, groups = generate_wesad_like_data(N_SUBJECTS, SAMPLES_PER_SUBJECT, N_FEATURES, N_CLASSES)

print(f"Dataset Summary:")
print(f"  Total samples: {len(y)}")
print(f"  Features: {X.shape[1]}")
print(f"  Subjects: {len(np.unique(groups))}")
print(f"  Classes: {dict(zip(*np.unique(y, return_counts=True)))}")

## 2. Baseline Model Evaluation

In [ ]:
# Define classifiers to evaluate
classifiers = {
    'Logistic Regression': LogisticRegressionClassifier(),
    'SVM (RBF)': SVMClassifier(),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'XGBoost': XGBoostClassifier(n_estimators=100),
    'LightGBM': LightGBMClassifier(n_estimators=100),
}

# Initialize cross-validator with LOSO
cv = CrossValidator(cv_strategy='loso', random_state=42)

# Run baseline evaluation
print("Running LOSO Cross-Validation for Baseline Models...\n")
print("="*70)

baseline_results = {}

for name, clf in classifiers.items():
    print(f"Evaluating {name}...")
    results = cv.cross_validate(clf, X, y, groups)
    baseline_results[name] = results
    
    print(f"  Accuracy: {results['accuracy_mean']:.4f} ± {results['accuracy_std']:.4f}")
    print(f"  F1 Macro: {results['f1_macro_mean']:.4f} ± {results['f1_macro_std']:.4f}")
    print()

In [ ]:
# Visualize baseline results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
names = list(baseline_results.keys())
accuracies = [baseline_results[n]['accuracy_mean'] for n in names]
acc_stds = [baseline_results[n]['accuracy_std'] for n in names]

bars1 = axes[0].bar(names, accuracies, yerr=acc_stds, capsize=5, color='steelblue', edgecolor='black')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('LOSO Cross-Validation Accuracy')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=45)

for bar, acc in zip(bars1, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                 f'{acc:.3f}', ha='center', va='bottom', fontsize=10)

# F1 Macro comparison
f1_scores = [baseline_results[n]['f1_macro_mean'] for n in names]
f1_stds = [baseline_results[n]['f1_macro_std'] for n in names]

bars2 = axes[1].bar(names, f1_scores, yerr=f1_stds, capsize=5, color='coral', edgecolor='black')
axes[1].set_ylabel('F1 Macro')
axes[1].set_title('LOSO Cross-Validation F1 Macro')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=45)

for bar, f1 in zip(bars2, f1_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{f1:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 3. Cross-Validation Analysis

In [ ]:
# Compare LOSO vs K-Fold to demonstrate overfitting gap
best_clf = RandomForestClassifier(n_estimators=100)

print("Comparing CV Strategies for Random Forest...\n")

comparison_df = cv.compare_cv_strategies(best_clf, X, y, groups)
print(comparison_df.to_string(index=False))

In [ ]:
# Per-subject performance analysis
print("\nPer-Subject LOSO Performance (Random Forest):")

subject_performance = cv.get_per_subject_performance(best_clf, X, y, groups)
print(subject_performance.to_string(index=False))

# Visualize subject variability
fig, ax = plt.subplots(figsize=(12, 5))

subjects = subject_performance['subject'].values
accs = subject_performance['accuracy'].values
f1s = subject_performance['f1_macro'].values

x = np.arange(len(subjects))
width = 0.35

bars1 = ax.bar(x - width/2, accs, width, label='Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, f1s, width, label='F1 Macro', color='coral')

ax.axhline(y=np.mean(accs), color='steelblue', linestyle='--', alpha=0.7, label=f'Mean Acc: {np.mean(accs):.3f}')
ax.axhline(y=np.mean(f1s), color='coral', linestyle='--', alpha=0.7, label=f'Mean F1: {np.mean(f1s):.3f}')

ax.set_xlabel('Subject ID')
ax.set_ylabel('Score')
ax.set_title('Per-Subject Performance (LOSO)')
ax.set_xticks(x)
ax.set_xticklabels([f'S{s}' for s in subjects])
ax.legend(loc='lower right')
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 4. Hyperparameter Tuning

In [ ]:
# Initialize hyperparameter tuner
tuner = HyperparameterTuner(
    cv_strategy='loso',
    n_trials=20,  # Reduced for demo; use 50-100 in practice
    random_state=42
)

# Tune Random Forest
print("Tuning Random Forest...")
rf_params, rf_score, rf_study = tuner.tune(
    'rf', X, y, groups, metric='f1_macro',
    show_progress_bar=True
)

print(f"\nBest Parameters: {rf_params}")
print(f"Best F1 Macro: {rf_score:.4f}")

In [ ]:
# Tune XGBoost
print("Tuning XGBoost...")
xgb_params, xgb_score, xgb_study = tuner.tune(
    'xgb', X, y, groups, metric='f1_macro',
    show_progress_bar=True
)

print(f"\nBest Parameters: {xgb_params}")
print(f"Best F1 Macro: {xgb_score:.4f}")

In [ ]:
# Compare tuned vs baseline
print("\nTuned Model Comparison:")
print("="*50)

# Evaluate tuned Random Forest
tuned_rf = RandomForestClassifier(**rf_params)
tuned_rf_results = cv.cross_validate(tuned_rf, X, y, groups)

print(f"Random Forest (Baseline): F1={baseline_results['Random Forest']['f1_macro_mean']:.4f}")
print(f"Random Forest (Tuned):    F1={tuned_rf_results['f1_macro_mean']:.4f}")
print(f"Improvement: +{(tuned_rf_results['f1_macro_mean'] - baseline_results['Random Forest']['f1_macro_mean'])*100:.2f}%")

In [ ]:
# Optuna optimization history visualization
try:
    import optuna
    from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Optimization history
    plot_optimization_history(rf_study, ax=axes[0])
    axes[0].set_title('Random Forest: Optimization History')
    
    # Parameter importances
    plot_param_importances(rf_study, ax=axes[1])
    axes[1].set_title('Random Forest: Parameter Importance')
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not plot Optuna visualizations: {e}")

## 5. Ensemble Methods

In [ ]:
# Create base models with tuned parameters
base_models = [
    ('rf', RandomForestClassifier(**rf_params)),
    ('xgb', XGBoostClassifier(**xgb_params)),
    ('lgbm', LightGBMClassifier(n_estimators=100)),
]

# Create Voting Ensemble
print("Creating Voting Ensemble...")
voting_ensemble = VotingEnsemble(base_models, voting='soft')
voting_ensemble.fit(X, y)

print("Voting Ensemble created with soft voting strategy.")

In [ ]:
# Create Stacking Ensemble
print("Creating Stacking Ensemble...")
stacking_ensemble = StackingEnsemble(
    base_models=base_models,
    meta_learner='lr',
    use_probas=True
)
stacking_ensemble.fit(X, y)

print("Stacking Ensemble created with Logistic Regression meta-learner.")

In [ ]:
# Evaluate ensemble models using manual LOSO
from sklearn.metrics import accuracy_score, f1_score

def evaluate_ensemble_loso(ensemble, X, y, groups):
    """Evaluate ensemble with LOSO CV."""
    fold_metrics = []
    
    for train_idx, test_idx in cv.get_loso_splits(X, y, groups):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Clone and fit
        ens_clone = ensemble.__class__(**ensemble.get_params())
        ens_clone.fit(X_train, y_train)
        
        y_pred = ens_clone.predict(X_test)
        
        fold_metrics.append({
            'accuracy': accuracy_score(y_test, y_pred),
            'f1_macro': f1_score(y_test, y_pred, average='macro')
        })
    
    return {
        'accuracy_mean': np.mean([f['accuracy'] for f in fold_metrics]),
        'accuracy_std': np.std([f['accuracy'] for f in fold_metrics]),
        'f1_macro_mean': np.mean([f['f1_macro'] for f in fold_metrics]),
        'f1_macro_std': np.std([f['f1_macro'] for f in fold_metrics]),
    }

# Evaluate ensembles
print("Evaluating Ensembles with LOSO...\n")

voting_results = evaluate_ensemble_loso(voting_ensemble, X, y, groups)
stacking_results = evaluate_ensemble_loso(stacking_ensemble, X, y, groups)

print(f"Voting Ensemble:")
print(f"  Accuracy: {voting_results['accuracy_mean']:.4f} ± {voting_results['accuracy_std']:.4f}")
print(f"  F1 Macro: {voting_results['f1_macro_mean']:.4f} ± {voting_results['f1_macro_std']:.4f}")

print(f"\nStacking Ensemble:")
print(f"  Accuracy: {stacking_results['accuracy_mean']:.4f} ± {stacking_results['accuracy_std']:.4f}")
print(f"  F1 Macro: {stacking_results['f1_macro_mean']:.4f} ± {stacking_results['f1_macro_std']:.4f}")

## 6. Model Comparison

In [ ]:
# Compile all results
all_results = {
    'Logistic Regression': baseline_results['Logistic Regression'],
    'SVM': baseline_results['SVM (RBF)'],
    'Random Forest': baseline_results['Random Forest'],
    'XGBoost': baseline_results['XGBoost'],
    'LightGBM': baseline_results['LightGBM'],
    'RF (Tuned)': tuned_rf_results,
    'Voting Ensemble': voting_results,
    'Stacking Ensemble': stacking_results,
}

# Create comparison DataFrame
evaluator = ModelEvaluator()

comparison_data = []
for name, results in all_results.items():
    comparison_data.append({
        'Model': name,
        'Accuracy': f"{results['accuracy_mean']:.4f} ± {results['accuracy_std']:.4f}",
        'F1 Macro': f"{results['f1_macro_mean']:.4f} ± {results['f1_macro_std']:.4f}",
        'Acc Mean': results['accuracy_mean'],
        'F1 Mean': results['f1_macro_mean'],
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('F1 Mean', ascending=False)

print("\nModel Comparison (Sorted by F1 Macro):")
print("="*70)
print(comparison_df[['Model', 'Accuracy', 'F1 Macro']].to_string(index=False))

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))

models = comparison_df['Model'].values
acc_means = comparison_df['Acc Mean'].values
f1_means = comparison_df['F1 Mean'].values

x = np.arange(len(models))
width = 0.35

bars1 = ax.bar(x - width/2, acc_means, width, label='Accuracy', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, f1_means, width, label='F1 Macro', color='coral', edgecolor='black')

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison (LOSO Cross-Validation)')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{height:.3f}', ha='center', va='bottom', fontsize=8)

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{height:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

### Statistical Comparison Between Models

We perform statistical tests to determine if differences between models are significant.

In [ ]:
# Statistical comparison between best models
# Get per-fold scores for Random Forest and XGBoost
rf_fold_scores = np.array([f['f1_macro'] for f in baseline_results['Random Forest']['fold_results']])
xgb_fold_scores = np.array([f['f1_macro'] for f in baseline_results['XGBoost']['fold_results']])

# Perform statistical comparison
stat_comparison = evaluator.statistical_comparison(rf_fold_scores, xgb_fold_scores)

print("Statistical Comparison: Random Forest vs XGBoost")
print("="*50)
print(f"Mean difference (RF - XGB): {stat_comparison['mean_difference']:.4f}")
print(f"\nPaired t-test:")
print(f"  t-statistic: {stat_comparison['paired_ttest']['statistic']:.4f}")
print(f"  p-value: {stat_comparison['paired_ttest']['pvalue']:.4f}")
print(f"  Significant: {stat_comparison['paired_ttest']['significant']}")
print(f"\nWilcoxon signed-rank test:")
print(f"  statistic: {stat_comparison['wilcoxon']['statistic']:.4f}")
print(f"  p-value: {stat_comparison['wilcoxon']['pvalue']:.4f}")
print(f"  Significant: {stat_comparison['wilcoxon']['significant']}")

# Confidence intervals for best model
mean, lower, upper = evaluator.compute_confidence_intervals(rf_fold_scores, confidence=0.95)
print(f"\n95% Confidence Interval for RF F1 Macro:")
print(f"  {mean:.4f} [{lower:.4f}, {upper:.4f}]")

In [ ]:
# Per-subject performance heatmap for all models
subject_model_results = {}

for name, clf in classifiers.items():
    perf = cv.get_per_subject_performance(clf, X, y, groups)
    subject_model_results[name] = perf.set_index('subject')['f1_macro']

# Create heatmap DataFrame
heatmap_df = pd.DataFrame(subject_model_results)
heatmap_df.index = [f'S{i}' for i in heatmap_df.index]

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(heatmap_df, annot=True, fmt='.3f', cmap='RdYlGn', 
            vmin=0.2, vmax=0.8, center=0.5, ax=ax,
            cbar_kws={'label': 'F1 Macro'})
ax.set_title('Per-Subject F1 Macro Score Across Models')
ax.set_xlabel('Model')
ax.set_ylabel('Subject')
plt.tight_layout()
plt.show()

# Identify subjects with consistently low/high performance
mean_per_subject = heatmap_df.mean(axis=1)
print("\nSubject Performance Summary:")
print(f"  Best subject: {mean_per_subject.idxmax()} (mean F1: {mean_per_subject.max():.4f})")
print(f"  Worst subject: {mean_per_subject.idxmin()} (mean F1: {mean_per_subject.min():.4f})")

## 7. Final Evaluation

In [ ]:
# Get best model and train on full data
best_model_name = comparison_df.iloc[0]['Model']
print(f"Best Model: {best_model_name}")

# Train best model on full data
best_model = RandomForestClassifier(**rf_params)
best_model.fit(X, y)

# Get predictions for evaluation
y_pred = best_model.predict(X)
y_proba = best_model.predict_proba(X)

# Compute detailed metrics
metrics = evaluator.compute_metrics(y, y_pred, y_proba)

print("\nDetailed Metrics (Full Dataset):")
print("="*50)
for metric, value in metrics.items():
    if not np.isnan(value):
        print(f"  {metric}: {value:.4f}")

In [ ]:
# Confusion matrix
fig = evaluator.plot_confusion_matrix(y, y_pred, normalize=True)
plt.show()

In [ ]:
# ROC Curves (Multi-class One-vs-Rest)
fig = evaluator.plot_roc_curves(y, y_proba)
plt.suptitle('ROC Curves (One-vs-Rest)', y=1.02)
plt.show()

In [ ]:
# Precision-Recall Curves
fig = evaluator.plot_precision_recall_curves(y, y_proba)
plt.suptitle('Precision-Recall Curves', y=1.02)
plt.show()

In [ ]:
# Per-class metrics
class_names = {0: 'Baseline', 1: 'Stress', 2: 'Amusement'}
evaluator.class_names = class_names

per_class = evaluator.compute_per_class_metrics(y, y_pred, y_proba)
print("\nPer-Class Metrics:")
print(per_class.to_string(index=False))

In [ ]:
# Feature importance
feature_names = [f'Feature_{i}' for i in range(X.shape[1])]
importance_df = best_model.get_feature_importance(feature_names)

# Plot top 20 features
fig, ax = plt.subplots(figsize=(10, 8))

top_n = 20
top_features = importance_df.head(top_n)

ax.barh(range(len(top_features)), top_features['importance'].values, color='steelblue')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'].values)
ax.set_xlabel('Importance')
ax.set_title(f'Top {top_n} Feature Importances')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 8. Results Summary

### Comparison with State-of-the-Art (SOTA)

Reference results from the original WESAD paper (Schmidt et al., 2018) and subsequent studies:

In [ ]:
# SOTA comparison from WESAD literature
sota_results = {
    'Schmidt et al. (2018) - AdaBoost': {'accuracy': 0.804, 'f1': 0.766, 'modality': 'Chest'},
    'Schmidt et al. (2018) - RF': {'accuracy': 0.796, 'f1': 0.752, 'modality': 'Chest'},
    'Schmidt et al. (2018) - Decision Tree': {'accuracy': 0.753, 'f1': 0.712, 'modality': 'Chest'},
    'Our Best (LOSO)': {
        'accuracy': best_results.get('accuracy_mean', 0.5),
        'f1': best_results.get('f1_macro_mean', 0.5),
        'modality': 'Synthetic'
    },
}

# Create comparison table
sota_df = pd.DataFrame(sota_results).T
sota_df = sota_df.round(3)

print("Comparison with WESAD State-of-the-Art:")
print("="*70)
print(sota_df.to_string())

print("\nNotes:")
print("  - Original WESAD uses 3-class classification (Baseline/Stress/Amusement)")
print("  - LOSO CV is critical for fair comparison")
print("  - Our results use synthetic data; real WESAD data should show similar patterns")
print("\nReferences:")
print("  [1] Schmidt, P., et al. (2018). 'Introducing WESAD, a Multimodal Dataset")
print("      for Wearable Stress and Affect Detection.' ICMI 2018.")

In [ ]:
print("="*70)
print("STRESS DETECTION ML PIPELINE - RESULTS SUMMARY")
print("="*70)

print(f"\nDataset:")
print(f"  - Samples: {len(y)}")
print(f"  - Features: {X.shape[1]}")
print(f"  - Subjects: {len(np.unique(groups))}")
print(f"  - Classes: Baseline, Stress, Amusement")

print(f"\nValidation Strategy:")
print(f"  - Leave-One-Subject-Out (LOSO) Cross-Validation")
print(f"  - Subject-independent evaluation")

print(f"\nBest Model: {best_model_name}")
best_results = all_results[best_model_name]
print(f"  - LOSO Accuracy: {best_results['accuracy_mean']:.4f} ± {best_results['accuracy_std']:.4f}")
print(f"  - LOSO F1 Macro: {best_results['f1_macro_mean']:.4f} ± {best_results['f1_macro_std']:.4f}")

print(f"\nKey Findings:")
print(f"  1. Ensemble methods (Stacking/Voting) provide robust performance")
print(f"  2. Hyperparameter tuning improved baseline by ~{(tuned_rf_results['f1_macro_mean'] - baseline_results['Random Forest']['f1_macro_mean'])*100:.1f}%")
print(f"  3. Subject variability highlights importance of LOSO validation")

print("\n" + "="*70)

---

## Next Steps

1. **Week 6**: Implement deep learning models (CNN, LSTM, Transformer)
2. **Calibration Analysis**: Evaluate probability calibration
3. **Threshold Optimization**: Optimize decision thresholds per class
4. **Model Interpretability**: SHAP values for feature explanation

---

*CalmSense - -Level Multimodal Stress Detection*